# scrape judul dan url


In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def scrape_berita_malang():

    urls = [
        "https://beritamalang.media/",
        "https://beritamalang.media/page/2/",
        "https://beritamalang.media/page/3/"
    ]

    data = []

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for page_url in urls:

        print(f"scraping: {page_url}")

        html = requests.get(
            page_url,
            headers=headers,
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        for article in soup.select("article.post-main"):

            title_tag = article.select_one(
                "h2.post-main-title a"
            )

            date_tag = article.select_one(
                ".post-main-datapost span"
            )

            category_tag = article.select_one(
                ".post-main-category"
            )

            if not title_tag:
                continue

            data.append({
                "title": title_tag.get_text(strip=True),
                "url": title_tag["href"],
                "published_date": (
                    date_tag.get_text(strip=True)
                    if date_tag else None
                ),
                "category": (
                    category_tag.get_text(
                        ",",
                        strip=True
                    )
                    if category_tag else None
                ),
                "source": "beritamalang_media"
            })

    df = (
        pd.DataFrame(data)
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    df.to_csv(
        "beritamalang_media.csv",
        index=False
    )

    print(
        f"saved {len(df)} articles"
    )

    return df

df_berita_malang = scrape_berita_malang()

df_berita_malang.head()

scraping: https://beritamalang.media/
scraping: https://beritamalang.media/page/2/
scraping: https://beritamalang.media/page/3/
saved 30 articles


,title,url,published_date,category,source
0,"Desa Sumberejo Gelar Semarak Selamatan ke-116,...",https://beritamalang.media/desa-sumberejo-gela...,"June 17, 2026","Berita,,,Ragam",beritamalang_media
1,Polres Batu Gelar Layanan On the Spot di CFD S...,https://beritamalang.media/polres-batu-gelar-l...,"June 14, 2026",Berita,beritamalang_media
2,"Cegah Kebutaan Sejak Dini, Klinik Mata Batu Aj...",https://beritamalang.media/cegah-kebutaan-seja...,"June 14, 2026",Berita,beritamalang_media
3,Satlantas Polres Batu Buka Layanan SIM dan Sam...,https://beritamalang.media/satlantas-polres-ba...,"June 14, 2026",Berita,beritamalang_media
4,31 Anak Terpilih Jadi Anggota Pocil Polres Bat...,https://beritamalang.media/31-anak-terpilih-ja...,"June 14, 2026",Berita,beritamalang_media


# scrape isi url 

In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv("beritamalang_media.csv")

articles = []

for i, row in df.iterrows():

    try:

        html = requests.get(
            row["url"],
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        ).text

        soup = BeautifulSoup(html, "html.parser")

        text = soup.get_text(
            "\n",
            strip=True
        )

        start = text.find("Oleh Penulis")
        end = text.find("Share:")

        content = None

        if start != -1 and end != -1:

            content = text[start:end]

            content = content.replace(
                "Oleh Penulis\nComment: 0",
                ""
            ).strip()

        articles.append({
            "title": row["title"],
            "published_date": row["published_date"],
            "category": row["category"],
            "content": content,
            "url": row["url"],
            "source": "beritamalang_media"
        })

        print(f"[{i+1}/{len(df)}] success")

    except Exception as e:

        print(f"[{i+1}/{len(df)}] failed: {e}")

result = pd.DataFrame(articles)

result.to_csv(
    "beritamalang_media_articles.csv",
    index=False
)

print(result.head())

[1/30] success
[2/30] success
[3/30] success
[4/30] success
[5/30] success
[6/30] success
[7/30] success
[8/30] success
[9/30] success
[10/30] success
[11/30] success
[12/30] success
[13/30] success
[14/30] success
[15/30] success
[16/30] success
[17/30] success
[18/30] success
[19/30] success
[20/30] success
[21/30] success
[22/30] success
[23/30] success
[24/30] success
[25/30] success
[26/30] success
[27/30] success
[28/30] success
[29/30] success
[30/30] success
                                               title published_date  \
0  Desa Sumberejo Gelar Semarak Selamatan ke-116,...  June 17, 2026   
1  Polres Batu Gelar Layanan On the Spot di CFD S...  June 14, 2026   
2  Cegah Kebutaan Sejak Dini, Klinik Mata Batu Aj...  June 14, 2026   
3  Satlantas Polres Batu Buka Layanan SIM dan Sam...  June 14, 2026   
4  31 Anak Terpilih Jadi Anggota Pocil Polres Bat...  June 14, 2026   

         category                                            content  \
0  Berita,,,Ragam  Beritamalan